# Using Machine Learning techniques to predict oral cancer

## Imports

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np

## Load in data, basic data information

In [ ]:
path = kagglehub.dataset_download("menegidio/oral-cancer-prediction-dataset")

file_path = os.path.join(path, "oral_cancer_prediction_dataset.csv")
df = pd.read_csv(file_path)

df.head()

In [ ]:
df.info()

## Data cleaning

In [ ]:
df.columns

### Dropping columns

In this stage we get rid of features that are synonymous with cancer (eg. Tumor Size, Cancer Stage).

In [ ]:
df = df.drop(df.columns[17:-2], axis=1)
df.head()

### Checking for NaN values

We check if there are any values missing from the dataset.
- `df.isna()` – returns a DataFrame of True/False values (True if the value in the original DataFrame is NaN, False otherwise)  
- `df.isna().any()` – for each column, checks if it contains at least one True value  
- `df.isna().any().any()` – returns True if there is **at least one NaN** in the original DataFrame, False otherwise

In [ ]:
df.isna().any().any()

There are no NaN values as is, so we may proceed.

### Checking data for unique values

Before modifying the data to be fit for training we should check how many unique values each feature can have, so that we can determine how to represent it (eg. use 0 and 1 for yes-no columns or one-hot encoding for those with more posiible values).

In [ ]:
for column in df.columns:
    print(f"{column} values: {df[column].unique()}")

### Modifying the data

In [ ]:
# Binary to 1/0
df["Gender"] = df["Gender"].map({"Male": 1, "Female": 0})

for column in df.columns:
    if np.isin("Yes", df[column].unique()):
        df[column] = df[column].map({"Yes": 1, "No": 0})

# One-Hot Encoding
df = pd.get_dummies(df, columns=["Country", "Diet (Fruits & Vegetables Intake)"])

# Convert all bool columns from OHE to int
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

df.head()

In [ ]:
# Move label to the rightmost side of the dataframe
cols = [c for c in df.columns if c != "Oral Cancer (Diagnosis)"] + ["Oral Cancer (Diagnosis)"]
df = df[cols]

### Renaming the columns

In this step we standardize the naming convention of features in the dataframe.

In [ ]:
def get_renamed_column(name):
    # Remove parentheses and their contents
    name = name[:name.index("(")] + name[name.index(")") + 1:] if "(" in name else name
    name = name.strip()
    name = name.lower()
    name = name.replace(" ", "_")
    name = name.replace("__", "_")

    return name


def get_rename_dict(column_names):
    rename_dict = {}
    for name in column_names:
        rename_dict[name] = get_renamed_column(name)

    return rename_dict

In [ ]:
df = df.rename(columns=get_rename_dict(df.columns))
df.head()